# ast_serialize

## Example 1: parse a source string

In [ ]:
# Example 1: parse returns (AST bytes, errors, type-ignores, import bytes, data); the hash is SHA-1 of the source
import hashlib
import ast_serialize

source = "import os\nfrom typing import List\nx: List[int] = [1]\n"
tree, errors, ignores, imports, data = ast_serialize.parse("m.py", source)
assert isinstance(tree, bytes) and len(tree) > 0 and isinstance(imports, bytes) and errors == [] and ignores == []
assert data["source_hash"] == hashlib.sha1(source.encode()).hexdigest() and data["is_partial_package"] is False
assert ast_serialize.parse("m.py", source)[0] == tree  # the same source gives the same bytes
print("Example 1:", len(tree), "AST bytes,", len(imports), "import bytes, data keys", sorted(data))

## Example 2: syntax errors come back as data, not as an exception

In [ ]:
# Example 2: a broken file still returns a (partial) tree plus a list of blocking errors with line and column
import ast_serialize

tree, errors, ignores, imports, data = ast_serialize.parse("bad.py", "def f(:\n    pass\n")
assert errors and all(error["blocker"] for error in errors), errors
assert errors[0]["line"] == 1 and errors[0]["column"] == 7 and "parameter" in errors[0]["message"], errors[0]
print("Example 2:", [(e["line"], e["column"], e["message"]) for e in errors])

## Example 3: `type: ignore` codes, `# mypy:` comments and template strings are reported

In [ ]:
# Example 3: the ignore list names line + codes; inline mypy config and t-strings (Python 3.14) are flagged in `data`
import ast_serialize

source = "x: int = 'a'  # type: ignore[assignment, misc]\n# mypy: disallow-untyped-defs\ndef f(a): return t'{a}'\n"
tree, errors, ignores, imports, data = ast_serialize.parse("m.py", source)
assert errors == [] and ignores == [(1, ["assignment", "misc"])], ignores
assert data["mypy_comments"] == [(2, "disallow-untyped-defs")] and data["uses_template_strings"] is True, data
print("Example 3: ignores", ignores, "| mypy comments", data["mypy_comments"])

## Example 4: reachability follows the platform and the Python version you name

In [ ]:
# Example 4: the same file gives different import metadata when the guarded branch is reachable and when it is not
import ast_serialize

on_platform = "import sys\nif sys.platform == 'win32':\n    import winreg\n"
on_version = "import sys\nif sys.version_info >= (3, 12):\n    import tomllib\n"
linux, windows = (ast_serialize.parse("m.py", on_platform, platform=p)[3] for p in ("linux", "win32"))
old, new = (ast_serialize.parse("m.py", on_version, python_version=v)[3] for v in ((3, 11), (3, 14)))
assert linux != windows and old != new and b"winreg" in linux and b"tomllib" in old
assert ast_serialize.parse("m.py", on_platform, platform="linux", always_true=["X"])[3] == linux
small = ast_serialize.parse("m.py", "def f():\n    return [i * i for i in range(10)]\n", skip_function_bodies=True)[0]
assert len(small) < len(ast_serialize.parse("m.py", "def f():\n    return [i * i for i in range(10)]\n")[0])
print("Example 4: platform and version change the import metadata; skip_function_bodies shrinks the tree")

## Example 5: the bytes read back into the same tree mypy's other parser builds

In [ ]:
# Example 5: mypy reads ast_serialize's bytes back into a tree (eager=True); it matches the stdlib-ast parser's tree
import ast_serialize, mypy.errors, mypy.options, mypy.parse

source = "class Point:\n    x: int\n    def norm(self) -> float:\n        return abs(self.x)\n\nORIGIN = Point()\n"
native, classic = mypy.options.Options(), mypy.options.Options()
classic.native_parser = False
native_tree = mypy.parse.parse(source, "m.py", "m", mypy.errors.Errors(native), native, eager=True)
classic_tree = mypy.parse.parse(source, "m.py", "m", mypy.errors.Errors(classic), classic)
native_defs, classic_defs = [str(d) for d in native_tree.defs], [str(d) for d in classic_tree.defs]
assert native.native_parser and native_defs and native_defs == classic_defs, (native_defs, classic_defs)
assert [type(d).__name__ for d in native_tree.defs] == ["ClassDef", "AssignmentStmt"]
type_bytes = ast_serialize.parse_type_string("list[int]", (1, 0, 1, 9))
assert isinstance(type_bytes, bytes) and b"list" in type_bytes and b"int" in type_bytes
print("Example 5: native tree == classic tree:", [type(d).__name__ for d in native_tree.defs])